In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

import joblib

In [2]:
df = pd.read_csv("../Dataset/final_dataset.csv")

df.head()

,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,dropout,avg_quiz_score,assessments_completed,avg_assessment_weight,avg_submission_day,video_clicks,login_frequency,avg_activity_day,registration_days
0,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,0,78.0,1.0,10.0,18.0,934.0,196.0,102.132653,270.0
1,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,0,70.0,1.0,10.0,22.0,1435.0,430.0,86.993023,270.0
2,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,1,0.0,0.0,0.0,0.0,281.0,76.0,2.355263,104.0
3,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,0,72.0,1.0,10.0,17.0,2158.0,663.0,106.147813,270.0
4,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,0,69.0,1.0,10.0,26.0,1034.0,352.0,91.934659,270.0


In [ ]:
# Select Features for KNN
features = [
    "avg_quiz_score",
    "assessments_completed",
    "avg_assessment_weight",
    "avg_submission_day",
    "video_clicks",
    "login_frequency",
    "avg_activity_day"
]

X = df[features]

In [ ]:
# Normalize Data
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(32593, 7)


In [ ]:
# Train KNN Model
knn = NearestNeighbors(
    n_neighbors=6,
    metric="euclidean"
)

knn.fit(X_scaled)

print("KNN Model Trained Successfully")

KNN Model Trained Successfully


In [6]:
# save the KNN model
joblib.dump(knn,"../backend/models/knn_model.pkl")

joblib.dump(scaler,"../backend/models/knn_scaler.pkl")

print("Model Saved")

Model Saved


In [ ]:
# Find Similar Students
student_index = 50

distance, indices = knn.kneighbors(
    [X_scaled[student_index]]
)

print(indices)

[[   50   368   727 28834 14916   226]]


In [ ]:
# Show Similar Students
similar_students = df.iloc[indices[0]]

similar_students

,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,dropout,avg_quiz_score,assessments_completed,avg_assessment_weight,avg_submission_day,video_clicks,login_frequency,avg_activity_day,registration_days
50,AAA,2013J,F,South Region,HE Qualification,80-90%,35-55,0,60,N,Pass,0,61.0,1.0,10.0,19.0,5645.0,1620.0,127.147531,270.0
368,AAA,2013J,M,East Anglian Region,Lower Than A Level,60-70%,35-55,0,60,N,Distinction,0,75.0,1.0,10.0,14.0,5974.0,1562.0,132.238796,270.0
727,AAA,2014J,F,East Anglian Region,Lower Than A Level,40-50%,35-55,0,60,N,Pass,0,56.0,1.0,10.0,24.0,5956.0,1571.0,108.899427,270.0
28834,FFF,2014J,M,Scotland,Lower Than A Level,80-90%,0-35,0,90,N,Pass,0,76.0,1.0,12.5,21.0,5648.0,1570.0,122.210828,270.0
14916,DDD,2013J,F,South Region,Lower Than A Level,70-80%,35-55,0,60,N,Pass,0,49.0,1.0,10.0,25.0,5612.0,1626.0,112.741082,270.0
226,AAA,2013J,M,West Midlands Region,A Level or Equivalent,60-70%,55<=,0,120,N,Distinction,0,81.0,1.0,10.0,19.0,5680.0,1441.0,134.962526,270.0


In [9]:
# Recommend Learning Resources
def recommend_resources(student):

    recommendations=[]

    if student["avg_quiz_score"]<60:
        recommendations.append(
            "Practice additional quizzes"
        )

    if student["video_clicks"]<100:
        recommendations.append(
            "Watch remaining video lectures"
        )

    if student["login_frequency"]<20:
        recommendations.append(
            "Increase weekly login frequency"
        )

    if student["assessments_completed"]<5:
        recommendations.append(
            "Complete pending assignments"
        )

    if student["avg_activity_day"]<15:
        recommendations.append(
            "Spend more daily learning time"
        )

    return recommendations

In [10]:
# Test Recommendation
student=df.iloc[100]

print("Dropout :",student["dropout"])

recommendations=recommend_resources(student)

for r in recommendations:
    print("•",r)

Dropout : 0
• Practice additional quizzes
• Complete pending assignments


In [11]:
# Recommendation Using Similar Students
student_index=100

distance,indices=knn.kneighbors(
    [X_scaled[student_index]]
)

similar=df.iloc[indices[0]]

print(similar[[
    "avg_quiz_score",
    "video_clicks",
    "login_frequency",
    "dropout"
]])

       avg_quiz_score  video_clicks  login_frequency  dropout
100              51.0        1829.0            627.0        0
65               60.0        1939.0            697.0        0
304              56.0        1588.0            544.0        0
15218            61.0        1709.0            610.0        0
49               65.0        2121.0            631.0        0
435              67.0        1737.0            561.0        0


In [12]:
# Personalized Recommendation Engine
def personalized_recommendation(student_index):

    distance,indices=knn.kneighbors(
        [X_scaled[student_index]]
    )

    student=df.iloc[student_index]

    print("="*60)
    print("Student",student_index)
    print("Dropout Risk :",student["dropout"])
    print("="*60)

    rec=recommend_resources(student)

    print("\nRecommended Actions\n")

    for r in rec:
        print("✔",r)

    print("\nMost Similar Students")

    display(df.iloc[
        indices[0]
    ][[
        "avg_quiz_score",
        "video_clicks",
        "login_frequency",
        "dropout"
    ]])

In [13]:
# Run Recommendation
personalized_recommendation(200)

Student 200
Dropout Risk : 1

Recommended Actions

✔ Complete pending assignments

Most Similar Students


,avg_quiz_score,video_clicks,login_frequency,dropout
200,67.0,1359.0,297.0,1
61,65.0,1185.0,320.0,1
17449,67.0,751.0,305.0,0
323,62.0,967.0,238.0,0
701,68.0,802.0,283.0,0
14770,58.0,1159.0,408.0,1


In [14]:
# Save Recommendations
recommendation=[]

for i in range(len(df)):

    rec=recommend_resources(df.iloc[i])

    recommendation.append(", ".join(rec))

df["recommendation"]=recommendation

df.to_csv("../Dataset/recommendation_dataset.csv",index=False)

print("Recommendation Dataset Saved")

Recommendation Dataset Saved
